In [1]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from sentence_transformers import SentenceTransformer
import os
import pickle
import faiss

loader = PyPDFDirectoryLoader("./sources")
docs=None
#check if docs.pkl exists
if os.path.exists("docs.pkl"):
    with open("docs.pkl", "rb") as f:
        docs = pickle.load(f)
else:
    docs = loader.load()
    docs = [i.page_content for i in docs]
    with open("docs.pkl", "wb") as f:
        pickle.dump(docs, f)
print("loaded docs")

loaded docs


In [2]:
class TextIndexer:
    def __init__(self, docs):
        self.docs = docs

    def gen_index(self, model_name):
        model_file_name = model_name.replace("/", "_")
        if os.path.exists(f"{model_file_name}.index"):
            print("Loading index from file")
            index = faiss.read_index(f"{model_file_name}.index")
            return index
        print("Generating index")
        model = SentenceTransformer(model_name)
        embeddings = model.encode(self.docs)
        index = faiss.IndexFlatL2(embeddings.shape[1])
        index.add(embeddings)
        faiss.write_index(index, f"{model_file_name}.index")
        return index
    
    def query_index(self, query, index, model_name, top_k=5):
        model = SentenceTransformer(model_name)
        query_embedding = model.encode([query])
        D, I = index.search(query_embedding, top_k)
        #should return results of the string
        return [self.docs[i] for i in I[0]]
       


In [3]:
text_indexer = TextIndexer(docs)
test_query = {
    "scan": "Replace the i-th element of the vector x with the minimum value from indices 0 through i. Use OpenMP to compute in parallel.",
    "reduce": "Return the value of the smallest odd number in the vector x. Use OpenMP to compute in parallel.",
    "dense": "Multiply the matrix A by the matrix B. Store the results in the matrix C. A is an MxK matrix, B is a KxN matrix, and C is a MxN matrix. The matrices are stored in row-major. Use OpenMP to compute in parallel.",
    "fft": "Compute the fourier transform of x in-place. Return the imaginary conjugate of each value. Use OpenMP to compute in parallel.",
    "geometry": "Return the perimeter of the smallest convex polygon that contains all the points in the vector points. Use OpenMP to compute in parallel.",
    "graph": "Count the number of connected components in the undirected graph defined by the adjacency matrix A. A is an NxN adjacency matrix stored in row-major. A is an undirected graph. Use OpenMP to compute in parallel.",
    "histogram": "Count the number of doubles in the vector x that have a fractional part in [0, 0.25), [0.25, 0.5), [0.5, 0.75), and [0.75, 1). Store the counts in `bins`. Use OpenMP to compute in parallel.",
    "search": "Return true if the vector x contains the value `target`. Return false otherwise. Use OpenMP to search in parallel.",
    "sort": "Sort vector of Result structs by start time in ascending order. Use OpenMP to sort in parallel.",
    "sparse": "Compute y = alpha*A*x + beta*y where alpha and beta are scalars, x and y are vectors, and A is a sparse matrix stored in COO format. A has dimensions MxN, x has N values, and y has M values. Use OpenMP to parallelize.",
    "stencil": "Set every cell's value to 1 if it has exactly one neighbor that's a 1. Otherwise set it to 0. Note that we only consider neighbors and not input_{i,j} when computing output_{i,j}. input and output are NxN grids of ints in row-major. Use OpenMP to compute in parallel.",
    "transform": "In the vector x negate the odd values and divide the even values by 2. Use OpenMP to compute in parallel.",
}


In [4]:
def run_test(model_name, indexer, query, dir, top_k=5, print_results=False):
    index = indexer.gen_index(model_name)
    print("Done generating index for model: " + model_name)
    print("-"*100)
    print("Querying index for: " + query)
    results = indexer.query_index(query, index, model_name, top_k=top_k)
    if print_results:
        print(*results) 
    #save results to file:
    model_file_name = model_name.replace("/", "_")
    with open(f"{dir}/{model_file_name}_no_prob_results.txt", "w") as f:
        for i in range(len(results)):
            f.write(f"Result {i+1}:\n")
            f.write(results[i])
            f.write("\n")
            f.write("-" * 100)
            f.write("\n")

#run_test("paraphrase-MiniLM-L6-v2", text_indexer, test_query)

In [5]:
list_of_models = [
    "avsolatorio/GIST-small-Embedding-v0",
    "thenlper/gte-large",
    "WhereIsAI/UAE-Large-V1",
    "avsolatorio/GIST-Embedding-v0",
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-base-en-v1.5",
    "BAAI/bge-large-en-v1.5",

    #These models are big, run if you have VRAM
    #"GritLM/GritLM-7B",     
    #"Salesforce/SFR-Embedding-Mistral",

]

for i in list_of_models:
    for dir, query in test_query.items():
        try:        
            run_test(i, text_indexer, query, dir)
        except Exception as e:
            print(f"Error with model: {i}")
            print(e)
            print("-"*100)
            continue

Loading index from file
Done generating index for model: avsolatorio/GIST-small-Embedding-v0
----------------------------------------------------------------------------------------------------
Querying index for: Replace the i-th element of the vector x with the minimum value from indices 0 through i. Use OpenMP to compute in parallel.
Error with model: avsolatorio/GIST-small-Embedding-v0
[Errno 2] No such file or directory: 'scan/avsolatorio_GIST-small-Embedding-v0_no_prob_results.txt'
----------------------------------------------------------------------------------------------------
Loading index from file
Done generating index for model: avsolatorio/GIST-small-Embedding-v0
----------------------------------------------------------------------------------------------------
Querying index for: Return the value of the smallest odd number in the vector x. Use OpenMP to compute in parallel.
Error with model: avsolatorio/GIST-small-Embedding-v0
[Errno 2] No such file or directory: 'redu

No sentence-transformers model found with name WhereIsAI/UAE-Large-V1. Creating a new one with MEAN pooling.


Error with model: thenlper/gte-large
[Errno 2] No such file or directory: 'transform/thenlper_gte-large_no_prob_results.txt'
----------------------------------------------------------------------------------------------------
Loading index from file
Done generating index for model: WhereIsAI/UAE-Large-V1
----------------------------------------------------------------------------------------------------
Querying index for: Replace the i-th element of the vector x with the minimum value from indices 0 through i. Use OpenMP to compute in parallel.


No sentence-transformers model found with name WhereIsAI/UAE-Large-V1. Creating a new one with MEAN pooling.


Error with model: WhereIsAI/UAE-Large-V1
[Errno 2] No such file or directory: 'scan/WhereIsAI_UAE-Large-V1_no_prob_results.txt'
----------------------------------------------------------------------------------------------------
Loading index from file
Done generating index for model: WhereIsAI/UAE-Large-V1
----------------------------------------------------------------------------------------------------
Querying index for: Return the value of the smallest odd number in the vector x. Use OpenMP to compute in parallel.


...

Error with model: WhereIsAI/UAE-Large-V1
[Errno 2] No such file or directory: 'sort/WhereIsAI_UAE-Large-V1_no_prob_results.txt'
----------------------------------------------------------------------------------------------------
Loading index from file
Done generating index for model: WhereIsAI/UAE-Large-V1
----------------------------------------------------------------------------------------------------
Querying index for: Compute y = alpha*A*x + beta*y where alpha and beta are scalars, x and y are vectors, and A is a sparse matrix stored in COO format. A has dimensions MxN, x has N values, and y has M values. Use OpenMP to parallelize.


No sentence-transformers model found with name WhereIsAI/UAE-Large-V1. Creating a new one with MEAN pooling.


Error with model: WhereIsAI/UAE-Large-V1
[Errno 2] No such file or directory: 'sparse/WhereIsAI_UAE-Large-V1_no_prob_results.txt'
----------------------------------------------------------------------------------------------------
Loading index from file
Done generating index for model: WhereIsAI/UAE-Large-V1
----------------------------------------------------------------------------------------------------
Querying index for: Set every cell's value to 1 if it has exactly one neighbor that's a 1. Otherwise set it to 0. Note that we only consider neighbors and not input_{i,j} when computing output_{i,j}. input and output are NxN grids of ints in row-major. Use OpenMP to compute in parallel.


No sentence-transformers model found with name WhereIsAI/UAE-Large-V1. Creating a new one with MEAN pooling.


Error with model: WhereIsAI/UAE-Large-V1
[Errno 2] No such file or directory: 'stencil/WhereIsAI_UAE-Large-V1_no_prob_results.txt'
----------------------------------------------------------------------------------------------------
Loading index from file
Done generating index for model: WhereIsAI/UAE-Large-V1
----------------------------------------------------------------------------------------------------
Querying index for: In the vector x negate the odd values and divide the even values by 2. Use OpenMP to compute in parallel.
Error with model: WhereIsAI/UAE-Large-V1
[Errno 2] No such file or directory: 'transform/WhereIsAI_UAE-Large-V1_no_prob_results.txt'
----------------------------------------------------------------------------------------------------
Loading index from file
Done generating index for model: avsolatorio/GIST-Embedding-v0
----------------------------------------------------------------------------------------------------
Querying index for: Replace the i-th el